# Нормирование Парето через `log` и оценка хвостового индекса

Распределение Парето с минимумом `x_min > 0` и хвостовым индексом `α > 0`:

$$\bar F(x) = \Pr(X > x) = \left(\frac{x_{\min}}{x}\right)^{\alpha},\quad x \ge x_{\min}.$$

Плотность: $f(x) = \alpha\, x_{\min}^{\alpha} / x^{\alpha+1}$.

**Ключевое наблюдение.** Если $X$ распределено по Парето с параметрами $(x_{\min},\alpha)$, то

$$Y \;:=\; \log\!\left(\tfrac{X}{x_{\min}}\right) \;\sim\; \mathrm{Exp}(\alpha).$$

Проверка: $\Pr(Y > y) = \Pr(X > x_{\min} e^y) = (x_{\min}/(x_{\min}e^y))^{\alpha} = e^{-\alpha y}$. Так что лог-нормирование переводит тяжелохвостовое распределение в лёгкохвостовое экспоненциальное.

**Следствие** (оценка Хилла / MLE).  $\mathbb E[Y] = 1/\alpha$, поэтому

$$\hat\alpha_{\mathrm{Hill}} \;=\; \frac{1}{\bar Y} \;=\; \frac{n}{\sum_{i=1}^n \log(X_i / x_{\min})}.$$

(В некоторых источниках — особенно в экстремальной теории — этот же параметр обозначают $k$ вместо $\alpha$; смысл один.)

Стандартная ошибка для $\hat\alpha$: $\mathrm{SE}(\hat\alpha) \approx \hat\alpha / \sqrt{n}$ (асимптотический результат для MLE экспоненциального распределения).

Ниже всё проверяем численно.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=42)

# True parameters
alpha_true = 2.5      # tail index (could be called k)
x_min = 1.0           # location / lower bound
n = 5000              # sample size

# Pareto via inverse CDF: U ~ U(0,1) ⇒ X = x_min · (1 - U)^(-1/α)
u = rng.uniform(size=n)
X = x_min * (1 - u) ** (-1.0 / alpha_true)

print(f"Min, max of X:        {X.min():.3f}, {X.max():.3f}")
print(f"Median of X:          {np.median(X):.3f}")
print(f"Mean of X (theory):   {alpha_true / (alpha_true - 1) * x_min:.3f}   (α/(α−1)·x_min)")
print(f"Mean of X (sample):   {X.mean():.3f}")
print()
print(f"Variance is finite iff α > 2.   Here α = {alpha_true}, so yes.")
print(f"Sample variance:      {X.var():.3f}")

## 1. Тяжёлый хвост Парето в исходной шкале

На линейной оси `X` сильно скошен, выборка "подавляется" редкими, но огромными значениями. Чтобы хвост был виден, используем лог-лог-шкалу для эмпирической выживаемости.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram on linear scale.
axes[0].hist(X[X < np.quantile(X, 0.99)], bins=60, color="steelblue", alpha=0.8)
axes[0].set_xlabel("X")
axes[0].set_ylabel("count")
axes[0].set_title("Гистограмма X в линейной шкале (99-й процентиль)")

# Empirical survival on log-log scale.
X_sorted = np.sort(X)
surv = 1 - np.arange(1, n + 1) / n
# Drop the rightmost point (survival = 0, log is -∞).
ax = axes[1]
ax.loglog(X_sorted[:-1], surv[:-1], "o", ms=2, alpha=0.5, label="эмпирическая $\\bar F$")
# Theoretical line: survival ∝ x^(-α).
xx = np.linspace(x_min, X_sorted[-2], 100)
ax.loglog(xx, (x_min / xx) ** alpha_true, "r--", lw=2,
           label=f"$(x_{{\\min}}/x)^{{{alpha_true}}}$")
ax.set_xlabel("x (log scale)")
ax.set_ylabel("$\\bar F(x)$ (log scale)")
ax.set_title("Эмпирическая хвостовая функция (log-log)")
ax.legend()
plt.tight_layout(); plt.show()

Прямая линия с наклоном `−α` в лог-лог-координатах — фирменный знак степенного хвоста.

## 2. Лог-нормирование: Парето → экспоненциальное

Применяем $Y = \log(X / x_{\min})$. По теории это должно быть $\mathrm{Exp}(\alpha)$, то есть плотность $\alpha e^{-\alpha y}$ на $[0,\infty)$.

In [ ]:
Y = np.log(X / x_min)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram of Y, with the Exp(α_true) PDF overlaid.
axes[0].hist(Y, bins=60, density=True, color="seagreen", alpha=0.7,
             label="гистограмма Y")
yy = np.linspace(0, Y.max(), 200)
axes[0].plot(yy, alpha_true * np.exp(-alpha_true * yy), "r-", lw=2,
             label=f"Exp($\\alpha$={alpha_true}) PDF")
axes[0].set_xlabel("Y = log(X / x_min)")
axes[0].set_ylabel("плотность")
axes[0].set_title("Y после лог-нормирования (должно быть Exp(α))")
axes[0].legend()

# Q-Q plot of Y vs theoretical Exp(α_true).
(osm, osr), (slope, intercept, r) = stats.probplot(
    Y, dist=stats.expon, sparams=(0, 1.0 / alpha_true), plot=axes[1])
axes[1].set_title(f"Q–Q plot: Y против Exp(α={alpha_true})")
axes[1].set_xlabel("теоретические квантили")
axes[1].set_ylabel("эмпирические квантили")
plt.tight_layout(); plt.show()

print(f"Q–Q regression slope:     {slope:.4f}   (ожидаем ≈ 1)")
print(f"Q–Q regression intercept: {intercept:.4f}  (ожидаем ≈ 0)")
print(f"Q–Q R²:                   {r**2:.5f}   (ожидаем ≈ 1)")

## 3. Оценка $\hat\alpha$ (оценка Хилла / MLE)

Так как $Y \sim \mathrm{Exp}(\alpha)$ имеет среднее $1/\alpha$, оценка максимального правдоподобия для $\alpha$:

$$\hat\alpha = \frac{1}{\bar Y} = \frac{n}{\sum_i \log(X_i/x_{\min})}.$$

Это совпадает с **оценкой Хилла** (Hill 1975), которая является каноническим методом оценки хвостового индекса.

Асимптотика: $\sqrt n(\hat\alpha - \alpha) \stackrel{d}{\to} \mathcal N(0, \alpha^2)$, то есть стандартная ошибка $\hat\alpha/\sqrt n$.

In [ ]:
alpha_hat = 1.0 / Y.mean()
se_hat = alpha_hat / np.sqrt(n)
ci95 = (alpha_hat - 1.96 * se_hat, alpha_hat + 1.96 * se_hat)

print(f"True α:                 {alpha_true}")
print(f"Hill / MLE estimate:    α̂ = {alpha_hat:.4f}")
print(f"Standard error:         SE(α̂) ≈ {se_hat:.4f}")
print(f"95% confidence interval: [{ci95[0]:.4f}, {ci95[1]:.4f}]")
print(f"Inside CI:              {ci95[0] <= alpha_true <= ci95[1]}")

## 4. Сходимость по выборке

Покажем, как $\hat\alpha$ сходится к $\alpha$ с ростом $n$. Заодно проверим, что фактическая дисперсия согласуется с асимптотической формулой $\alpha^2/n$.

In [ ]:
ns = np.array([50, 100, 200, 500, 1000, 2000, 5000, 10000])
n_replicas = 400

alpha_hats = np.empty((len(ns), n_replicas))
for i, ni in enumerate(ns):
    u_block = rng.uniform(size=(n_replicas, ni))
    X_block = x_min * (1 - u_block) ** (-1.0 / alpha_true)
    Y_block = np.log(X_block / x_min)
    alpha_hats[i] = 1.0 / Y_block.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(alpha_true, color="red", lw=2, label=f"истинное α = {alpha_true}")
ax.errorbar(ns, alpha_hats.mean(axis=1),
            yerr=alpha_hats.std(axis=1),
            fmt="o", capsize=4, color="steelblue",
            label="среднее α̂ ± std (из 400 реплик)")
# Theoretical asymptotic SE: α/√n.
se_theory = alpha_true / np.sqrt(ns)
ax.fill_between(ns, alpha_true - 1.96 * se_theory, alpha_true + 1.96 * se_theory,
                alpha=0.15, color="red", label="95% полоса (α/√n)")
ax.set_xscale("log")
ax.set_xlabel("размер выборки n")
ax.set_ylabel("оценка α̂")
ax.set_title("Сходимость оценки Хилла к истинному α")
ax.legend()
plt.tight_layout(); plt.show()

print("   n     empirical SE     theoretical α/√n")
for ni, row in zip(ns, alpha_hats):
    print(f"  {ni:5d}     {row.std():.4f}            {alpha_true/np.sqrt(ni):.4f}")

## 5. Робастность: выбор порога $x_{\min}$

В реальных данных мы редко знаем $x_{\min}$ точно. Стандартная практика — оценивать $\alpha$ только по "хвосту" $X_{(k)} \le X_{(k+1)} \le \dots$ выше выбранного порога. Это даёт **оценку Хилла с порогом** (Hill estimator with threshold):

$$\hat\alpha_k \;=\; \frac{k}{\sum_{i=1}^{k} \log(X_{(n-i+1)} / X_{(n-k)})}.$$

(`X_(n-k)` — это `(n-k)`-я порядковая статистика, играющая роль эмпирического $x_{\min}$.)

Построим "Hill plot": $\hat\alpha_k$ как функция $k$.

In [ ]:
X_sorted = np.sort(X)  # ascending
ks = np.arange(20, n)
alpha_k = np.empty_like(ks, dtype=float)
for j, k in enumerate(ks):
    # Top-k order statistics: X_sorted[-k:]; threshold X_sorted[-k-1].
    top = X_sorted[-k:]
    threshold = X_sorted[-k - 1]
    alpha_k[j] = k / np.sum(np.log(top / threshold))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(alpha_true, color="red", lw=2, label=f"истинное α = {alpha_true}")
ax.plot(ks, alpha_k, color="steelblue", lw=1)
ax.fill_between(ks,
                alpha_k - 1.96 * alpha_k / np.sqrt(ks),
                alpha_k + 1.96 * alpha_k / np.sqrt(ks),
                alpha=0.2, color="steelblue", label="95% CI")
ax.set_xlabel("k (число порядковых статистик с правого хвоста)")
ax.set_ylabel("$\\hat\\alpha_k$")
ax.set_title("Hill plot — стабильное плато даёт надёжную оценку")
ax.legend()
plt.tight_layout(); plt.show()

## 6. Насколько это не Гаусс?

Подгоним к данным $X$ нормальное распределение $\mathcal N(\hat\mu, \hat\sigma^2)$ и сравним по нескольким метрикам.

**Финитность моментов.** У Парето $k$-й момент $\mathbb E[X^k]$ конечен только при $\alpha > k$. У гауссиана конечны *все* моменты. Для нашего $\alpha = 2.5$:

| момент | Парето $\alpha = 2.5$ | гауссиан |
|---|---|---|
| среднее   $(k{=}1)$ | финитно | финитно |
| дисперсия $(k{=}2)$ | финитно, но "раздутое" | финитна |
| **скошенность** $(k{=}3)$ | $\infty$ (теор., при $\alpha\le 3$) | $0$ |
| **эксцесс** $(k{=}4)$ | $\infty$ (теор., при $\alpha\le 4$) | $0$ |

Эмпирические значения скошенности/эксцесса будут огромными и крайне неустойчивыми — это и есть структурная неотличимость Парето от гауссиана.

In [ ]:
# Moments comparison
mu_g, sigma_g = X.mean(), X.std()

print(f"Pareto α = {alpha_true}  —  финитность моментов:")
print(f"  E[X]      финитно при α > 1:  {alpha_true > 1}")
print(f"  Var(X)    финитно при α > 2:  {alpha_true > 2}")
print(f"  Skewness  финитно при α > 3:  {alpha_true > 3}  ← у нас НЕТ")
print(f"  Kurtosis  финитно при α > 4:  {alpha_true > 4}  ← у нас НЕТ")
print()
print(f"Эмпирические значения (могут сильно скакать от запуска к запуску):")
print(f"  mean      X.mean() = {X.mean():.4f}")
print(f"  std       X.std()  = {X.std():.4f}")
print(f"  skewness            = {stats.skew(X):.4f}      (Gauss: 0)")
print(f"  excess kurtosis     = {stats.kurtosis(X):.4f}    (Gauss: 0)")

In [ ]:
# Q–Q plot against best-fit Gaussian — should curve sharply in the right tail.
fig, ax = plt.subplots(figsize=(7.5, 5))
stats.probplot(X, dist=stats.norm, sparams=(mu_g, sigma_g), plot=ax)
ax.set_title(f"Q–Q plot: X против $\\mathcal{{N}}({mu_g:.2f}, {sigma_g:.2f}^2)$")
ax.set_xlabel("квантили гауссиана")
ax.set_ylabel("квантили выборки")
ax.get_lines()[1].set_linewidth(2)
plt.tight_layout(); plt.show()

In [ ]:
# Tail probabilities: Pareto vs Gaussian fit, vs empirical.
fig, ax = plt.subplots(figsize=(9, 5))
xx = np.linspace(max(mu_g, x_min) * 1.01, np.quantile(X, 0.9995), 400)
surv_pareto = (x_min / xx) ** alpha_true
surv_gauss = stats.norm.sf(xx, loc=mu_g, scale=sigma_g)

X_sorted = np.sort(X)
emp_s = 1 - np.arange(1, n + 1) / n

ax.semilogy(xx, surv_pareto, "r-", lw=2, label=f"Парето ($\\alpha={alpha_true}$)")
ax.semilogy(xx, surv_gauss, "b--", lw=2, label=f"$\\mathcal{{N}}({mu_g:.2f}, {sigma_g:.2f}^2)$")
ax.semilogy(X_sorted[:-1], emp_s[:-1], "k.", ms=2, alpha=0.3, label="эмпирическое")
ax.set_xlabel("x"); ax.set_ylabel("$\\bar F(x)$  (log)")
ax.set_title("Хвостовая функция: Парето против гауссиана с тем же $\\hat\\mu, \\hat\\sigma$")
ax.set_ylim(1e-12, 1)
ax.legend()
plt.tight_layout(); plt.show()

print("Отношение хвостовых вероятностей на разных уровнях:")
for k_sigma in [1, 2, 3, 4, 5, 6]:
    x_k = mu_g + k_sigma * sigma_g
    if x_k > x_min:
        sp = (x_min / x_k) ** alpha_true
        sg = stats.norm.sf(x_k, loc=mu_g, scale=sigma_g)
        ratio = sp / sg if sg > 0 else float('inf')
        print(f"  P(X > μ + {k_sigma}σ):  Парето = {sp:.2e},  гауссиан = {sg:.2e},  отношение = {ratio:.1e}×")

In [ ]:
# Statistical tests for normality.
from scipy.stats import shapiro, anderson, kstest

X_sub = X[:5000] if n > 5000 else X
stat_sw, p_sw = shapiro(X_sub)
print(f"Shapiro–Wilk on X:     stat = {stat_sw:.6f},  p-value = {p_sw:.3e}")
print(f"  → {'отклоняем' if p_sw < 0.05 else 'НЕ отклоняем'} гипотезу гауссовости на 5%")

a_d = anderson(X, dist="norm")
print(f"\nAnderson–Darling on X: stat = {a_d.statistic:.4f}")
print(f"  Критические значения для α-уровней {tuple(a_d.significance_level)}:")
print(f"  {a_d.critical_values}")
print(f"  → отклоняем во всех {'(stat >> max crit value)' if a_d.statistic > a_d.critical_values.max() else ''}")

ks_stat, ks_p = kstest(X, lambda v: stats.norm.cdf(v, loc=mu_g, scale=sigma_g))
print(f"\nKolmogorov–Smirnov on X vs N(μ̂,σ̂²): stat = {ks_stat:.4f},  p = {ks_p:.3e}")

# For reference, the same test on Y (which is exponential, also not Gaussian).
Y_sub = Y[:5000] if n > 5000 else Y
stat_y, p_y = shapiro(Y_sub)
print(f"\nДля справки, Y = log(X/x_min) экспоненциальное, тоже не Gaussian:")
print(f"  Shapiro–Wilk on Y:   stat = {stat_y:.6f},  p-value = {p_y:.3e}")

In [ ]:
# How does the sample maximum scale?
n_max = n
gauss_max_typical = np.sqrt(2 * np.log(n_max))
pareto_max_typical = n_max ** (1.0 / alpha_true)

print(f"Типичный размер максимума выборки длины n = {n}:")
print(f"  Гауссиан N(0,1):   max ≈ √(2 log n) = {gauss_max_typical:.2f}  (в единицах σ)")
print(f"  Парето α = {alpha_true}:    max ≈ n^(1/α)  = {pareto_max_typical:.1f}  (в единицах x_min)")
print()
print(f"Эмпирический максимум:                X.max()   = {X.max():.2f}")
print(f"В единицах σ от среднего:             (X.max() − μ̂) / σ̂ = {(X.max() - mu_g) / sigma_g:.2f}σ")
tail_prob = stats.norm.sf((X.max() - mu_g) / sigma_g)
print(f"Гауссиан-вероятность такого отклонения: {tail_prob:.3e}")
print(f"  ← если бы X был гауссовским, такой максимум появлялся бы примерно")
print(f"     раз в {1/tail_prob:.1e} наблюдений.  У нас он есть в выборке из {n}.")

**Резюме количественной "не-гауссовости":**

| Метрика                       | Гаусс       | Парето $\alpha = 2.5$            |
|-------------------------------|-------------|----------------------------------|
| Skewness                      | $0$         | $\infty$ (теор.); эмп. большая   |
| Excess kurtosis               | $0$         | $\infty$                         |
| Q–Q правый хвост              | прямая      | резко вверх                       |
| $\Pr(X > \mu + 5\sigma)$      | $\sim 3\cdot 10^{-7}$ | в десятки–миллионы раз больше |
| Shapiro–Wilk p-value          | $\ge 0.05$  | $\approx 10^{-50}$               |
| Рост максимума с $n$          | $\sqrt{\log n}$ | $n^{1/\alpha}$               |

**Главный вывод.**  Гауссиан с теми же $\hat\mu, \hat\sigma$ занижает вероятности "редких" событий на много порядков величины — даже без формальных тестов это видно из Q–Q-плота и эмпирической хвостовой функции в лог-шкале.

Лог-нормирование $Y = \log(X/x_{\min})$ убирает основную часть этого расхождения (превращая Парето в экспоненциальное), но и экспоненциальное всё ещё *не* гауссовское — асимметрия и положительная скошенность остаются (см. p-value Shapiro–Wilk на $Y$).

## Итог

1. **Лог-нормирование** $Y = \log(X/x_{\min})$ переводит распределение Парето $(\alpha)$ в стандартное экспоненциальное $\mathrm{Exp}(\alpha)$. Это резко облегчает работу с тяжёлым хвостом: вместо степенной асимптотики мы имеем линейный закон убывания плотности.
2. **MLE / Хилл-оценка** хвостового индекса в одной формуле: $\hat\alpha = n / \sum_i \log(X_i/x_{\min})$.
3. **Асимптотика** $\sqrt n(\hat\alpha - \alpha) \to \mathcal N(0, \alpha^2)$ согласуется с экспериментом.
4. На практике порог $x_{\min}$ редко известен — используется Hill plot для выбора стабильного плато.

Замечание: то же $\hat\alpha$ можно получить и через регрессию в лог-лог координатах эмпирической выживаемости (`np.polyfit(log(x), log(survival), 1)`), но MLE-оценка асимптотически эффективнее.